# 07 - RAG v1 trên 30 câu dev (Kaggle / Colab)

**Mục đích.** Chạy RAG v1 (truy hồi hybrid BM25 + BGE-M3 hợp nhất bằng RRF, rồi Qwen2.5-7B-Instruct 4-bit sinh câu trả lời bằng prompt v1 có kiểm tra ký tự CJK và sinh lại) trên 30 câu dev của tập 100 câu miền Quy chế ĐHQGHN. Notebook phục vụ hai việc của Tuần 3:

| Việc theo kế hoạch | Sản phẩm của notebook |
|---|---|
| T2: RAG v1, ghi độ trễ và bộ nhớ GPU | `rag_v1_dev_results.json` (môi trường, cấu hình, độ trễ, VRAM, tổng hợp) |
| T3: chạy RAG v1 trên 30 câu dev để thử giao thức chấm ảo giác | `rag_v1_dev_answers.jsonl` (khóa `id`, `answer`), dùng làm đầu vào của `scripts/grading_tool.py export --split dev` |

**Phạm vi.**
- Chỉ dùng `gold_dev.jsonl`. Dataset không chứa `gold_test.jsonl`, và notebook kiểm tra điều này.
- Cấu hình truy hồi (`rrf_k = 60`, không xếp hạng lại theo tiêu đề) khớp retrieval v1 do B chốt trên `main` (`docs/results/retrieval_domain_dev.json`); top-5 là giá trị tạm cho đến khi có ablation k ∈ {3, 5, 10}. Nếu cấu hình đổi, sửa các hằng tương ứng ở mục 5; nếu bật xếp hạng lại theo tiêu đề thì phải sửa thêm lời gọi `answer_question` (tham số `doc_titles`) và `title_rerank` trong file kết quả.
- Notebook **không** chấm đúng/sai. Cột `auto_flag` chỉ cho biết câu trả lời có đúng bằng câu từ chối cố định hay không; cột `first_gold_rank` cho biết thứ hạng của đoạn nguồn đúng trong top-k. Nhãn ảo giác do người chấm gán theo `docs/hallucination_protocol.md` (có trong PR giao thức chấm ảo giác).
- Câu trả lời của 30 câu dev dùng để thử giao thức chấm, không đưa vào bảng kết quả của báo cáo như kết quả đánh giá cuối.

**Cách chạy trên Kaggle**
1. Tạo Dataset từ thư mục `vi-qa-rag-v1-dev-dataset/`. Thư mục này gồm `src/`, `scripts/build_index.py`, `scripts/run_eval.py`, `data/domain/` (chỉ PDF Quy chế), `data/gold/gold_dev.jsonl` và `data/splits/gold_split.json`; không nằm trong repo. Có thể dùng chung Dataset `vi-qa-rag-extractive` của notebook 08 (notebook tìm `src/generate.py` trong `/kaggle/input`; thư mục `checkpoint/` thừa không ảnh hưởng). Sinh từ commit đang chạy: `mkdir vi-qa-rag-v1-dev-dataset && git archive <commit> src scripts/build_index.py scripts/run_eval.py data/domain/dhqghn_quy_che_dao_tao_dai_hoc_2014.pdf data/gold/gold_dev.jsonl data/splits/gold_split.json | tar -x -C vi-qa-rag-v1-dev-dataset`.
2. Tạo Notebook mới, chọn *File > Import notebook* và nạp file này, rồi *Add Input* để gắn Dataset. *Session options > Accelerator*: **GPU T4 x2** (mã chỉ dùng GPU 0). *Internet*: **On**.
3. Chạy *Run All*. Lần đầu tải khoảng 15 GB trọng số Qwen và khoảng 2 GB trọng số BGE-M3.
4. Tải ba file trong `/kaggle/working/` (`rag_v1_dev_answers.jsonl`, `rag_v1_dev_results.json`, `rag_v1_dev_items.csv`) về và lưu vào `docs/results/rag_v1_dev/` trong repo.

## 1. Thiết lập môi trường

`CUDA_VISIBLE_DEVICES=0` ép chỉ dùng một GPU để số đo VRAM tương ứng một T4. `HF_HOME` trỏ vào ổ tạm của Kaggle để bộ nhớ đệm trọng số không làm đầy `/kaggle/working`.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if os.path.isdir("/kaggle/temp"):
    os.environ["HF_HOME"] = "/kaggle/temp/hf"

!pip install -q -U bitsandbytes accelerate
!pip install -q sentence-transformers rank-bm25 pdfplumber

In [ ]:
import csv
import glob
import hashlib
import importlib.metadata as md
import json
import platform
import shutil
import statistics
import sys
import time
from pathlib import Path

import torch
import transformers

assert torch.cuda.is_available(), "Không có GPU. Chọn Accelerator là GPU T4 trong Session options."

props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu_name": props.name,
    "gpu_total_gb": round(props.total_memory / 1e9, 2),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "accelerate": md.version("accelerate"),
    "bitsandbytes": md.version("bitsandbytes"),
    "sentence_transformers": md.version("sentence-transformers"),
    "rank_bm25": md.version("rank-bm25"),
    "pdfplumber": md.version("pdfplumber"),
}
print(json.dumps(ENV, indent=2))

## 2. Nạp mã nguồn và dữ liệu từ Dataset

`/kaggle/input` chỉ đọc, trong khi `build_index.py` ghi index vào `data/index/`, nên cả thư mục được sao sang ổ ghi được. Mã băm SHA-256 của từng file `.py` và của `gold_dev.jsonl` được ghi vào file kết quả để đối chiếu với commit trong repo.

In [ ]:
matches = glob.glob("/kaggle/input/**/src/generate.py", recursive=True) + glob.glob("./**/src/generate.py", recursive=True)
assert matches, "Không tìm thấy src/generate.py. Hãy gắn Dataset bằng 'Add Input' (Kaggle) hoặc tải thư mục dataset lên cạnh notebook (Colab)."
DATASET_ROOT = Path(matches[0]).resolve().parents[1]
print("dataset:", DATASET_ROOT)

# Đường dẫn tuyệt đối vì ô sau gọi os.chdir(REPO).
REPO = (Path("/kaggle/temp/repo") if Path("/kaggle/temp").is_dir() else Path("repo")).resolve()
OUT = (Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")).resolve()
# Luôn sao lại từ Dataset để lần chạy lại không dùng nhầm bản và index cũ.
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(DATASET_ROOT, REPO)

for required in ("src/generate.py", "src/pipeline.py", "src/retrieval.py", "scripts/build_index.py",
                 "scripts/run_eval.py", "data/domain/dhqghn_quy_che_dao_tao_dai_hoc_2014.pdf",
                 "data/gold/gold_dev.jsonl", "data/splits/gold_split.json"):
    assert (REPO / required).exists(), f"Dataset thiếu file: {required}"
assert not (REPO / "data" / "gold" / "gold_test.jsonl").exists(), "Dataset không được chứa gold_test.jsonl"

CODE_SNAPSHOT = {
    str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest()[:12]
    for p in sorted(REPO.glob("src/*.py")) + sorted(REPO.glob("scripts/*.py"))
}
GOLD_PATH = REPO / "data" / "gold" / "gold_dev.jsonl"
GOLD_SHA = hashlib.sha256(GOLD_PATH.read_bytes()).hexdigest()[:12]
print(json.dumps(CODE_SNAPSHOT, indent=2), "\ngold_dev sha256:", GOLD_SHA)

## 3. Dựng index cho kho Quy chế

Chạy đúng `scripts/build_index.py --corpus domain` của B. Số chunk in ra cần là 216 (kho hiện hành của nhóm).

In [ ]:
!cd {REPO} && python scripts/build_index.py --corpus domain 2>&1 | grep -v -i "it/s\|Batches" | tail -n 25

## 4. Nạp retriever, văn bản các đoạn và 30 câu dev

Mã đoạn có dạng `dieu21_khoan3`; một Khoản dài có thể bị tách thành nhiều chunk (`_diemX`, `_wN`), và mọi chunk đó đều tính là Khoản đúng. Ô này kiểm tra: đủ 30 câu (27 trong phạm vi, 3 ngoài phạm vi, theo kế hoạch mục 3.1), tập `id` trùng đúng `dev_ids` trong `data/splits/gold_split.json` (chống nhầm câu test vào dev), và mỗi câu trong phạm vi có chunk của Khoản đúng trong index.

In [ ]:
sys.path.insert(0, str(REPO))
os.chdir(REPO)

from scripts.run_eval import load_bm25, load_dense
from src.generate import SYSTEM_PROMPT, is_refusal, load_model, make_generate_fn
from src.pipeline import answer_question

bm25 = load_bm25("domain")
dense = load_dense("domain")
with open(REPO / "data" / "index" / "domain" / "chunks_meta.json", encoding="utf-8") as f:
    CHUNK_TEXTS = {item["id"]: item["text"] for item in json.load(f)}
print("chunks:", len(CHUNK_TEXTS))

with open(GOLD_PATH, encoding="utf-8-sig") as f:
    ITEMS = [json.loads(line) for line in f if line.strip()]
ITEMS.sort(key=lambda x: x["id"])
n_in = sum(bool(i["in_scope"]) for i in ITEMS)
print(len(ITEMS), "câu;", n_in, "trong phạm vi;", len(ITEMS) - n_in, "ngoài phạm vi")
assert len(ITEMS) == 30 and n_in == 27, "gold_dev.jsonl không khớp kỳ vọng 30 câu (27 trong phạm vi)"
assert all(i.get("split") == "dev" for i in ITEMS), "Có câu không thuộc split dev"
with open(REPO / "data" / "splits" / "gold_split.json", encoding="utf-8") as f:
    assert {i["id"] for i in ITEMS} == set(json.load(f)["dev_ids"]), "id trong gold_dev.jsonl không khớp dev_ids của gold_split.json"


def gold_prefix(item):
    return f"dieu{item['article']}_khoan{item['clause']}"


def is_gold_id(doc_id, prefix):
    return doc_id == prefix or doc_id.startswith(prefix + "_")


for item in ITEMS:
    if item["in_scope"]:
        assert item["clause"] is not None, f"{item['id']}: thiếu Khoản"
        assert any(is_gold_id(i, gold_prefix(item)) for i in CHUNK_TEXTS), f"{item['id']}: không có chunk của Khoản đúng"

## 5. Nạp Qwen2.5-7B-Instruct ở dạng 4-bit

Dùng `load_model` của `src/generate.py` (NF4, double quantization, compute dtype float16, GPU 0). Bộ nhớ GPU được ghi sau khi đã nạp cả BGE-M3 và Qwen, vì hai mô hình cùng nằm trên một T4 khi chạy demo.

In [ ]:
# Cấu hình chạy (một nơi duy nhất; được ghi vào file kết quả).
TOP_K, RRF_K, MAX_RETRIES, MAX_NEW_TOKENS, SEED = 5, 60, 2, 256, 0
torch.manual_seed(SEED)  # chỉ ảnh hưởng các lần sinh lại có lấy mẫu khi lẫn CJK

torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
model, tokenizer = load_model()
torch.cuda.synchronize()
LOAD = {
    "load_seconds": round(time.perf_counter() - t0, 1),
    "vram_allocated_after_load_gb": round(torch.cuda.memory_allocated() / 1e9, 2),
    "vram_peak_during_load_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
}
print(json.dumps(LOAD, indent=2))
generate_fn = make_generate_fn(model, tokenizer, max_new_tokens=MAX_NEW_TOKENS)

## 6. Chạy RAG v1 trên 30 câu dev

Mỗi câu chạy một lần, thứ tự theo `id`, không chỉnh prompt hay tham số giữa các câu. Câu khởi động (không tính vào số đo) được chạy trước để kernel CUDA và bộ nhớ đệm không làm lệch số đo của câu đầu.

`auto_flag`:
- `refused`: câu trả lời đúng bằng câu từ chối cố định;
- `answered`: các trường hợp còn lại (cần người chấm đọc);
- `first_gold_rank`: thứ hạng đầu tiên của chunk thuộc Khoản đúng trong danh sách truy hồi (`None` nếu không có, hoặc câu ngoài phạm vi).

In [ ]:
from src.pipeline import retrieve_contexts

# Khởi động cả Qwen và BGE-M3 (câu truy vấn nhúng lần đầu chậm hơn), không tính vào số đo.
_ = generate_fn([{"role": "user", "content": "Xin chào"}], {})
_ = retrieve_contexts(bm25, dense, CHUNK_TEXTS, "Khởi động", top_k=TOP_K, rrf_k=RRF_K)


def run_item(item):
    torch.cuda.reset_peak_memory_stats()
    res = answer_question(generate_fn, bm25, dense, CHUNK_TEXTS, item["question"],
                          top_k=TOP_K, rrf_k=RRF_K, max_retries=MAX_RETRIES)
    torch.cuda.synchronize()
    first_gold_rank = None
    if item["in_scope"]:
        ranks = [r for r, i in enumerate(res["sources"], start=1) if is_gold_id(i, gold_prefix(item))]
        first_gold_rank = min(ranks) if ranks else None
    return {
        "id": item["id"], "in_scope": bool(item["in_scope"]), "question": item["question"],
        "answer": res["answer"], "auto_flag": "refused" if is_refusal(res["answer"]) else "answered",
        "sources": res["sources"], "first_gold_rank": first_gold_rank,
        "invalid_citations": res["invalid_citations"],
        "cjk": res["cjk"], "attempts": res["attempts"],
        "retrieval_seconds": res["retrieval_seconds"], "generation_seconds": res["seconds"],
        "vram_peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }


ROWS = [run_item(item) for item in ITEMS]
for r in ROWS:
    print(f"[{r['id']}] {r['auto_flag']} | in_scope={r['in_scope']} | gold rank: {r['first_gold_rank']} | {r['sources']}")
    print("  Q:", r["question"])
    print("  A:", r["answer"].replace("\n", " "))

## 7. Tổng hợp và lưu kết quả

Ba file được ghi vào `/kaggle/working/`:
- `rag_v1_dev_answers.jsonl`: mỗi dòng `{"id", "answer"}`, đầu vào của `grading_tool.py export`;
- `rag_v1_dev_results.json`: môi trường, cấu hình, mã băm mã nguồn và gold, độ trễ, VRAM, từng câu;
- `rag_v1_dev_items.csv`: từng câu, dễ đọc trong bảng tính.

In [ ]:
in_rows = [r for r in ROWS if r["in_scope"]]
out_rows = [r for r in ROWS if not r["in_scope"]]
SUMMARY = {
    "n_items": len(ROWS), "n_in_scope": len(in_rows), "n_out_of_scope": len(out_rows),
    "in_scope_refused": sum(r["auto_flag"] == "refused" for r in in_rows),
    "out_of_scope_refused": sum(r["auto_flag"] == "refused" for r in out_rows),
    "cjk_final": sum(r["cjk"] for r in ROWS),
    "n_regenerated": sum(r["attempts"] > 1 for r in ROWS),
    "n_invalid_citations": sum(bool(r["invalid_citations"]) for r in ROWS),
    "gold_in_top_k": {f"top{k}": sum(r["first_gold_rank"] is not None and r["first_gold_rank"] <= k for r in in_rows)
                      for k in (1, 3, 5)},
    "median_retrieval_seconds": round(statistics.median(r["retrieval_seconds"] for r in ROWS), 3),
    "median_generation_seconds": round(statistics.median(r["generation_seconds"] for r in ROWS), 2),
    "max_generation_seconds": max(r["generation_seconds"] for r in ROWS),
    "max_vram_peak_gb": max(r["vram_peak_gb"] for r in ROWS),
}
print(json.dumps(SUMMARY, indent=2, ensure_ascii=False))

RESULTS = {
    "env": ENV, "code_snapshot": CODE_SNAPSHOT, "gold_dev_sha256_12": GOLD_SHA,
    "config": {
        "retrieval": "hybrid (BM25 + BGE-M3, RRF)", "top_k": TOP_K, "rrf_k": RRF_K, "title_rerank": False,
        "model_id": "Qwen/Qwen2.5-7B-Instruct (4-bit NF4)", "max_new_tokens": MAX_NEW_TOKENS,
        "max_retries": MAX_RETRIES, "seed": SEED, "n_chunks": len(CHUNK_TEXTS), "system_prompt": SYSTEM_PROMPT,
    },
    "load": LOAD, "summary": SUMMARY, "items": ROWS,
}
with open(OUT / "rag_v1_dev_results.json", "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, ensure_ascii=False, indent=2)

with open(OUT / "rag_v1_dev_answers.jsonl", "w", encoding="utf-8", newline="\n") as f:
    for r in ROWS:
        f.write(json.dumps({"id": r["id"], "answer": r["answer"]}, ensure_ascii=False) + "\n")

fields = ["id", "in_scope", "question", "answer", "auto_flag", "sources", "first_gold_rank", "invalid_citations",
          "cjk", "attempts", "retrieval_seconds", "generation_seconds", "vram_peak_gb"]
with open(OUT / "rag_v1_dev_items.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for r in ROWS:
        writer.writerow({**r, "sources": ";".join(r["sources"]),
                         "invalid_citations": ";".join(map(str, r["invalid_citations"]))})
print("saved:", sorted(p.name for p in OUT.glob("rag_v1_dev_*")))

## 8. Cách đọc kết quả

| Quan sát | Ý nghĩa | Việc tiếp theo |
|---|---|---|
| `out_of_scope_refused` < 3 | Có câu ngoài phạm vi không bị trả lời bằng câu từ chối cố định. `is_refusal` chỉ nhận đúng câu đó, nên câu từ chối diễn đạt khác cũng rơi vào đây | Người chấm đọc câu trả lời để quyết định `sai_bia` hay `tu_choi_dung`; ghi nhận cho ablation ràng buộc ngữ cảnh (Tuần 4 T2) |
| `in_scope_refused` > 0 | Từ chối trong khi văn bản có đáp án; ứng viên `tu_choi_sai` | Đọc `first_gold_rank` và các đoạn trong `sources`: `None` nghĩa là Khoản đúng không nằm trong top-k; có thứ hạng thì chưa kết luận được nguyên nhân (đoạn con có thể không chứa đáp án, hoặc mô hình từ chối) |
| `gold_in_top_k.top5` thấp so với 27 | Truy hồi bỏ sót Khoản đúng | Chuyển cho B (chốt retrieval v1 ở Tuần 3 T3) |
| `cjk_final` > 0 hoặc `n_regenerated` > 0 | Cơ chế sinh lại CJK được kích hoạt, hoặc không khử được | Ghi số lần vào báo cáo |
| `n_invalid_citations` > 0 | Câu trả lời trích dẫn nguồn `[n]` không có trong danh sách đoạn truy hồi | Cho người chấm biết khi chấm; giao thức chỉ quy định trích dẫn số Điều, Khoản sai là thông tin bịa, còn `[n]` ngoài danh sách chưa có quy tắc riêng |
| `max_vram_peak_gb` gần 15 | Qwen và BGE-M3 gần chiếm hết T4 | Giảm `TOP_K` hoặc chuyển BGE-M3 sang CPU khi chạy demo |

Sau khi tải kết quả về, bước tiếp theo nằm ngoài notebook: `python scripts/grading_tool.py export --split dev --system rag_v1=docs/results/rag_v1_dev/rag_v1_dev_answers.jsonl --out-dir <thư mục ngoài repo>` (thư mục chứa `key.json` không được commit).